# TRIMP de Bannister & Modèle Fitness-Fatigue de Coggan
## Tests, illustrations et extension data-driven

**Références principales**
- Banister et al. (1975). *A system model of training for athletic performance.* Aust. J. Sports Med., 7, 57–61.
- Morton, Fitz-Clarke & Banister (1990). *Modeling human performance in running.* J. Appl. Physiol., 69(3), 1171–1177.
- Edwards (1993). *The heart rate monitor book.* Polar Electro Oy.
- Lucia et al. (2003). *Tour de France versus Vuelta a España: which is harder?* Med. Sci. Sports Exerc., 35(5), 872–878.
- Allen & Coggan (2010). *Training and Racing with a Power Meter.* VeloPress.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit

# Style global
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})


---
## 1. TRIMP de Bannister (1975)

Le TRIMP (*Training IMPulse*) quantifie la charge interne d'une séance par :

$$\mathrm{TRIMP} = D \times \Delta \mathrm{HR} \times y$$

où :
- $D$ : durée de la séance (min)
- $\Delta \mathrm{HR} = \dfrac{\overline{\mathrm{HR}} - \mathrm{HR_{rest}}}{\mathrm{HR_{max}} - \mathrm{HR_{rest}}}$ : fraction de la réserve cardiaque utilisée
- $y$ : facteur de pondération non-linéaire lié à la lactatémie

$$y = a \cdot e^{b \cdot \Delta \mathrm{HR}}$$

avec, selon Banister (1991) :
- **Hommes** : $a = 0{,}64$, $b = 1{,}92$
- **Femmes** : $a = 0{,}86$, $b = 1{,}67$

*Source : Banister (1991), cité dans Morton et al. (1990) et Fellrnr.com/wiki/TRIMP.*


In [ ]:
def compute_delta_hr(hr_mean, hr_rest, hr_max):
    """Compute normalised heart rate reserve fraction."""
    return (hr_mean - hr_rest) / (hr_max - hr_rest)


def weighting_factor(delta_hr, sex="male"):
    """Compute Banister exponential weighting factor y.

    Parameters
    ----------
    delta_hr : float or array
        Normalised HR reserve fraction (0–1).
    sex : str
        'male' (a=0.64, b=1.92) or 'female' (a=0.86, b=1.67).
        Banister (1991), via Morton et al. (1990).
    """
    if sex == "male":
        a, b = 0.64, 1.92
    else:
        a, b = 0.86, 1.67
    return a * np.exp(b * delta_hr)


def trimp_banister(duration_min, hr_mean, hr_rest, hr_max, sex="male"):
    """Compute Banister TRIMP for a session.

    Banister et al. (1975); Banister (1991).
    """
    delta_hr = compute_delta_hr(hr_mean, hr_rest, hr_max)
    y = weighting_factor(delta_hr, sex=sex)
    return duration_min * delta_hr * y


# ── Illustration : facteur y en fonction de %HRR ──────────────────────────
dhr = np.linspace(0, 1, 300)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panneau gauche : courbe y(ΔHR)
ax = axes[0]
ax.plot(dhr * 100, weighting_factor(dhr, "male"),   label="Hommes (a=0.64, b=1.92)", color="#2e7bcf", lw=2)
ax.plot(dhr * 100, weighting_factor(dhr, "female"), label="Femmes (a=0.86, b=1.67)", color="#e05c3a", lw=2, ls="--")
ax.set_xlabel("%HRR")
ax.set_ylabel("Facteur de pondération y")
ax.set_title("Facteur y de Bannister selon %HRR")
ax.legend()

# Panneau droit : TRIMP d'une séance de 60 min selon HR moyenne
hr_rest, hr_max = 45, 190
hr_range = np.linspace(hr_rest + 5, hr_max - 5, 300)
dhr_range = compute_delta_hr(hr_range, hr_rest, hr_max)

ax2 = axes[1]
ax2.plot(hr_range, trimp_banister(60, hr_range, hr_rest, hr_max, "male"),
         label="Hommes", color="#2e7bcf", lw=2)
ax2.plot(hr_range, trimp_banister(60, hr_range, hr_rest, hr_max, "female"),
         label="Femmes", color="#e05c3a", lw=2, ls="--")
ax2.set_xlabel("FC moyenne (bpm)")
ax2.set_ylabel("TRIMP Bannister")
ax2.set_title("TRIMP Bannister — séance de 60 min\n(HRrest=45, HRmax=190)")
ax2.legend()

plt.tight_layout()
plt.savefig("fig_trimp_banister.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure sauvegardée : fig_trimp_banister.png")


### Exemple numérique

Athlète masculin : HRrest = 45 bpm, HRmax = 190 bpm.
- Séance 1 : 90 min à FC moyenne 140 bpm (sortie longue facile)
- Séance 2 : 45 min à FC moyenne 170 bpm (allure seuil)


In [ ]:
hr_rest, hr_max = 45, 190

seances = [
    {"label": "Sortie longue (90 min, 140 bpm)", "dur": 90, "hr": 140},
    {"label": "Seuil (45 min, 170 bpm)",         "dur": 45, "hr": 170},
]

print(f"{'Séance':<40} {'ΔHRR':>6} {'y':>6} {'TRIMP':>8}")
print("-" * 65)
for s in seances:
    dhr = compute_delta_hr(s["hr"], hr_rest, hr_max)
    y   = weighting_factor(dhr, "male")
    t   = trimp_banister(s["dur"], s["hr"], hr_rest, hr_max, "male")
    print(f"{s['label']:<40} {dhr:>6.3f} {y:>6.3f} {t:>8.1f}")


---
## 2. Évolutions : Edwards (1993) et Lucia et al. (2003)

### TRIMP d'Edwards (1993)

Somme pondérée du temps passé dans 5 zones définies en % FCmax :

| Zone | %FCmax   | Coefficient |
|------|----------|-------------|
| 1    | 50–60 %  | 1           |
| 2    | 60–70 %  | 2           |
| 3    | 70–80 %  | 3           |
| 4    | 80–90 %  | 4           |
| 5    | 90–100 % | 5           |

$$\mathrm{TRIMP_{Edwards}} = \sum_{z=1}^{5} t_z \times c_z$$

Les seuils et coefficients sont **arbitraires** — aucun ancrage physiologique validé.  
*Source : Edwards (1993). The heart rate monitor book. Polar Electro Oy.*

### TRIMP de Lucia et al. (2003)

Trois zones ancrées sur les **seuils ventilatoires** (SV1, SV2) :

| Zone | Délimitation | Coefficient |
|------|--------------|-------------|
| 1    | < SV1        | 1           |
| 2    | SV1–SV2      | 2           |
| 3    | > SV2        | 3           |

$$\mathrm{TRIMP_{Lucia}} = t_1 + 2\,t_2 + 3\,t_3$$

Les seuils ventilatoires sont physiologiquement fondés, mais les coefficients 1/2/3 restent  
arbitraires — il n'est pas prouvé que la zone 3 représente exactement 3× le stress de la zone 1.  
*Source : Lucia et al. (2003). Med. Sci. Sports Exerc., 35(5), 872–878.*


In [ ]:
EDWARDS_ZONES = [
    (0.50, 0.60, 1),
    (0.60, 0.70, 2),
    (0.70, 0.80, 3),
    (0.80, 0.90, 4),
    (0.90, 1.00, 5),
]

LUCIA_COEFS = [1, 2, 3]


def trimp_edwards(time_in_zones_min):
    """Compute Edwards TRIMP from time in 5 HR zones.

    Parameters
    ----------
    time_in_zones_min : list of 5 floats
        Minutes spent in zones 1–5.
    Edwards (1993).
    """
    coefs = [c for _, _, c in EDWARDS_ZONES]
    return sum(t * c for t, c in zip(time_in_zones_min, coefs))


def trimp_lucia(time_in_zones_min):
    """Compute Lucia TRIMP from time in 3 ventilatory zones.

    Parameters
    ----------
    time_in_zones_min : list of 3 floats
        Minutes spent below SV1, between SV1-SV2, above SV2.
    Lucia et al. (2003).
    """
    return sum(t * c for t, c in zip(time_in_zones_min, LUCIA_COEFS))


# ── Comparaison sur deux séances types ──────────────────────────────────────
# Séance A : endurance longue — 90 min
# Temps dans zones Edwards (approx. pour HR moy ~140, HRmax 190)
seance_A_edwards = [10, 30, 40, 10, 0]   # zones 1-5
seance_A_lucia   = [20, 60, 10]           # zones SV1-/SV1-SV2/SV2+

# Séance B : intervals seuil — 45 min
seance_B_edwards = [5, 5, 15, 15, 5]
seance_B_lucia   = [5, 20, 20]

print("Comparaison des méthodes TRIMP
")
print(f"{'Séance':<30} {'Bannister':>10} {'Edwards':>9} {'Lucia':>7}")
print("-" * 60)

# Bannister (calculé ci-dessus)
trimp_A_ban = trimp_banister(90, 140, 45, 190, "male")
trimp_B_ban = trimp_banister(45, 170, 45, 190, "male")

for label, ban, edw_z, luc_z in [
    ("Endurance longue (90 min)", trimp_A_ban, seance_A_edwards, seance_A_lucia),
    ("Seuil/intervals (45 min)",  trimp_B_ban, seance_B_edwards, seance_B_lucia),
]:
    edw = trimp_edwards(edw_z)
    luc = trimp_lucia(luc_z)
    print(f"{label:<30} {ban:>10.1f} {edw:>9.0f} {luc:>7.0f}")


# ── Figure : comparaison visuelle ────────────────────────────────────────────
methods  = ["Bannister", "Edwards", "Lucia"]
val_A    = [trimp_A_ban, trimp_edwards(seance_A_edwards), trimp_lucia(seance_A_lucia)]
val_B    = [trimp_B_ban, trimp_edwards(seance_B_edwards), trimp_lucia(seance_B_lucia)]

x = np.arange(len(methods))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars_A = ax.bar(x - width/2, val_A, width, label="Endurance longue (90 min)", color="#2e7bcf", alpha=0.85)
bars_B = ax.bar(x + width/2, val_B, width, label="Seuil/intervals (45 min)",  color="#e05c3a", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel("Score TRIMP")
ax.set_title("Comparaison Bannister / Edwards / Lucia\npour deux séances types")
ax.legend()
ax.bar_label(bars_A, fmt="%.0f", padding=3)
ax.bar_label(bars_B, fmt="%.0f", padding=3)
plt.tight_layout()
plt.savefig("fig_trimp_comparaison.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 3. Modèle fitness-fatigue de Morton, Fitz-Clarke & Banister (1990)

Chaque séance génère deux effets opposés à décroissance exponentielle :

$$g(t) = g(t-1)\,e^{-1/\tau_1} + w(t)\left(1 - e^{-1/\tau_1}\right)$$
$$h(t) = h(t-1)\,e^{-1/\tau_2} + w(t)\left(1 - e^{-1/\tau_2}\right)$$
$$\hat{p}(t) = p_0 + k_1\,g(t) - k_2\,h(t)$$

- $g(t)$ : composante positive (forme — *fitness*)
- $h(t)$ : composante négative (fatigue)
- $w(t)$ : charge du jour $t$ (TRIMP)
- $\tau_1 > \tau_2$ : la forme persiste plus longtemps que la fatigue
- $k_2 > k_1$ : la fatigue est d'amplitude immédiate plus grande que la forme

**Valeurs de référence** (Morton et al., 1990) :  
$k_1 = 1{,}0$, $k_2 = 1{,}8$–$2{,}0$, $\tau_1 = 49$–$50$ j, $\tau_2 = 11$ j.

*Source : Morton, Fitz-Clarke & Banister (1990). J. Appl. Physiol., 69(3), 1171–1177.*


In [ ]:
def simulate_fitness_fatigue(daily_trimp, k1=1.0, k2=2.0, tau1=50.0, tau2=11.0, p0=100.0):
    """Simulate Banister fitness-fatigue impulse-response model.

    Morton, Fitz-Clarke & Banister (1990).

    Parameters
    ----------
    daily_trimp : array-like
        Daily TRIMP values (one per day).
    k1, k2 : float
        Gain terms for fitness and fatigue.
    tau1, tau2 : float
        Time constants in days for fitness and fatigue.
    p0 : float
        Baseline performance.
    """
    n = len(daily_trimp)
    g = np.zeros(n)
    h = np.zeros(n)
    perf = np.zeros(n)

    decay1 = np.exp(-1.0 / tau1)
    decay2 = np.exp(-1.0 / tau2)

    for t, w in enumerate(daily_trimp):
        g_prev = g[t - 1] if t > 0 else 0.0
        h_prev = h[t - 1] if t > 0 else 0.0
        g[t] = g_prev * decay1 + w * (1 - decay1)
        h[t] = h_prev * decay2 + w * (1 - decay2)
        perf[t] = p0 + k1 * g[t] - k2 * h[t]

    return g, h, perf


# ── Scénario : bloc de charge + affûtage ────────────────────────────────────
np.random.seed(42)
n_days = 120

# Séquence réaliste : charge progressive + coupure + pic
trimp_seq = np.zeros(n_days)
# Jours 0-29 : base (TRIMP~40)
trimp_seq[:30]  = 40 + np.random.normal(0, 5, 30)
# Jours 30-59 : bloc intensif (TRIMP~70)
trimp_seq[30:60] = 70 + np.random.normal(0, 10, 30)
# Jours 60-89 : semaine de récup + nouveau bloc (TRIMP~50 puis 80)
trimp_seq[60:70] = 20 + np.random.normal(0, 5, 10)
trimp_seq[70:90] = 80 + np.random.normal(0, 10, 20)
# Jours 90-119 : affûtage progressif
trimp_seq[90:] = np.linspace(60, 15, 30) + np.random.normal(0, 5, 30)
trimp_seq = np.clip(trimp_seq, 0, None)

g, h, perf = simulate_fitness_fatigue(trimp_seq)
days = np.arange(n_days)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)

axes[0].bar(days, trimp_seq, color="#aac4e0", label="TRIMP journalier")
axes[0].set_ylabel("TRIMP")
axes[0].set_title("Charge journalière")
axes[0].legend()

axes[1].plot(days, g,    color="#2e7bcf", lw=2, label=f"Forme g(t)  [τ₁={50}j]")
axes[1].plot(days, h,    color="#e05c3a", lw=2, label=f"Fatigue h(t) [τ₂={11}j]")
axes[1].set_ylabel("Amplitude (u.a.)")
axes[1].set_title("Composantes fitness et fatigue")
axes[1].legend()

axes[2].plot(days, perf, color="#2c9c5a", lw=2, label="Performance estimée p(t)")
axes[2].axhline(100, color="gray", ls=":", lw=1, label="Niveau de base p₀")
axes[2].set_xlabel("Jours")
axes[2].set_ylabel("Performance (u.a.)")
axes[2].set_title("Performance prédite — Morton et al. (1990)")
axes[2].legend()

plt.tight_layout()
plt.savefig("fig_fitness_fatigue.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 4. Déclinaison de Coggan : CTL, ATL, TSB (Allen & Coggan, 2010)

Coggan simplifie le modèle de Banister en supprimant les gains $k_1, k_2$ pour  
obtenir des **moyennes exponentielles pondérées** directement interprétables :

$$\mathrm{CTL}(t) = \mathrm{CTL}(t-1)\,e^{-1/42} + \mathrm{TSS}(t)\left(1 - e^{-1/42}\right)$$
$$\mathrm{ATL}(t) = \mathrm{ATL}(t-1)\,e^{-1/7}  + \mathrm{TSS}(t)\left(1 - e^{-1/7}\right)$$
$$\mathrm{TSB}(t) = \mathrm{CTL}(t) - \mathrm{ATL}(t)$$

- **CTL** (*Chronic Training Load*) : forme de fond — constante de temps 42 j  
- **ATL** (*Acute Training Load*) : fatigue récente — constante de temps 7 j  
- **TSB** (*Training Stress Balance*) : "fraîcheur" = forme − fatigue

**TSS** (Training Stress Score) remplace le TRIMP en normalisant sur 1 heure à la puissance seuil.  
Repères pratiques (Allen & Coggan, 2010 ; Friel, cité dans TrainingPeaks) :  
TSB ∈ [+15, +25] → pic de forme optimal ; TSB < −30 → surcharge sévère.

*Source : Allen & Coggan (2010). Training and Racing with a Power Meter. VeloPress.*  
*Voir aussi : Coggan, A. The science of the TrainingPeaks Performance Manager. trainingpeaks.com (2023).*


In [ ]:
def simulate_ctl_atl_tsb(daily_tss, tau_ctl=42.0, tau_atl=7.0):
    """Simulate Coggan CTL/ATL/TSB performance management chart.

    Allen & Coggan (2010).

    Parameters
    ----------
    daily_tss : array-like
        Daily Training Stress Score.
    tau_ctl, tau_atl : float
        Time constants in days for CTL and ATL.
    """
    n = len(daily_tss)
    ctl = np.zeros(n)
    atl = np.zeros(n)
    tsb = np.zeros(n)

    decay_ctl = np.exp(-1.0 / tau_ctl)
    decay_atl = np.exp(-1.0 / tau_atl)

    for t, tss in enumerate(daily_tss):
        ctl_prev = ctl[t - 1] if t > 0 else 0.0
        atl_prev = atl[t - 1] if t > 0 else 0.0
        ctl[t] = ctl_prev * decay_ctl + tss * (1 - decay_ctl)
        atl[t] = atl_prev * decay_atl + tss * (1 - decay_atl)
        tsb[t] = ctl[t] - atl[t]

    return ctl, atl, tsb


# On réutilise la séquence précédente comme proxy TSS
ctl, atl, tsb = simulate_ctl_atl_tsb(trimp_seq)

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

axes[0].bar(days, trimp_seq, color="#aac4e0", alpha=0.6, label="TSS journalier")
axes[0].plot(days, ctl, color="#2e7bcf", lw=2.5, label=f"CTL — Forme (τ=42 j)")
axes[0].plot(days, atl, color="#e05c3a", lw=2.0, ls="--", label=f"ATL — Fatigue (τ=7 j)")
axes[0].set_ylabel("TSS / CTL / ATL")
axes[0].set_title("Performance Management Chart — Coggan (Allen & Coggan, 2010)")
axes[0].legend()

axes[1].fill_between(days, tsb, 0,
                     where=(tsb >= 0), color="#2c9c5a", alpha=0.35, label="TSB > 0 (fraîcheur)")
axes[1].fill_between(days, tsb, 0,
                     where=(tsb < 0),  color="#e05c3a", alpha=0.25, label="TSB < 0 (fatigue)")
axes[1].plot(days, tsb, color="#333", lw=1.5)
axes[1].axhline(0,   color="gray", ls=":", lw=1)
axes[1].axhline(15,  color="#2c9c5a", ls="--", lw=1, alpha=0.7, label="TSB = +15 (pic optimal)")
axes[1].axhline(-30, color="#c0392b", ls="--", lw=1, alpha=0.7, label="TSB = −30 (seuil surcharge)")
axes[1].set_xlabel("Jours")
axes[1].set_ylabel("TSB")
axes[1].set_title("Training Stress Balance (TSB = CTL − ATL)")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("fig_ctl_atl_tsb.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 5. Limites illustrées : la FC moyenne aveugle aux intervalles

La limite la plus connue du TRIMP de Bannister : deux séances de durée identique  
avec la **même FC moyenne** produisent le même TRIMP, alors que la séance fractionnée  
est physiologiquement bien plus stressante (lactate, perturbation neuromusculaire, etc.).

*Source : discuté dans Desgorces et al. (2007) et Veohtu.com (Solomon, 2020).*


In [ ]:
# Séance A : 60 min en continu à FC=145 (endurance aérobie)
# Séance B : 60 min avec 6x5min à FC=175 + récup 5min à FC=115
# FC moyenne dans les deux cas ≈ 145

hr_rest, hr_max = 45, 190
duration = 60

# Séance A : FC constante
hr_A = 145

# Séance B : FC variable (valeurs seconde par seconde simulées)
t_sec = np.arange(0, duration * 60)
# alternance 5min effort / 5min récup sur 60 min
period = 600  # 10 min par cycle (5 travail + 5 récup)
hr_B = np.where((t_sec % period) < 300, 175, 115).astype(float)
hr_B_mean = hr_B.mean()

# TRIMP Bannister avec FC instantanée (intégration)
def trimp_banister_continuous(hr_series, hr_rest, hr_max, sex="male"):
    """Compute TRIMP by integrating over a HR time series (1 value per second)."""
    dt_min = 1 / 60.0  # chaque point = 1 seconde = 1/60 min
    dhr = compute_delta_hr(hr_series, hr_rest, hr_max)
    y   = weighting_factor(dhr, sex)
    return np.sum(dhr * y * dt_min)

trimp_A = trimp_banister(duration, hr_A, hr_rest, hr_max, "male")
trimp_B_avg = trimp_banister(duration, hr_B_mean, hr_rest, hr_max, "male")
trimp_B_cont = trimp_banister_continuous(hr_B, hr_rest, hr_max, "male")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Profil FC
ax = axes[0]
ax.axhline(hr_A, color="#2e7bcf", lw=2, label=f"Séance A : FC constante ({hr_A} bpm)")
ax.plot(t_sec / 60, hr_B, color="#e05c3a", lw=1, alpha=0.7,
        label=f"Séance B : intervalles (moy. {hr_B_mean:.0f} bpm)")
ax.set_xlabel("Temps (min)")
ax.set_ylabel("FC (bpm)")
ax.set_title("Profils de fréquence cardiaque")
ax.legend(fontsize=9)

# Barres TRIMP
ax2 = axes[1]
labels = ["A (continu,
FC moy.)", "B (intervalles,
FC moy. seule)", "B (intervalles,
FC intégrée)"]
values = [trimp_A, trimp_B_avg, trimp_B_cont]
colors = ["#2e7bcf", "#e05c3a", "#c0392b"]
bars = ax2.bar(labels, values, color=colors, alpha=0.85)
ax2.bar_label(bars, fmt="%.1f", padding=4)
ax2.set_ylabel("TRIMP Bannister")
ax2.set_title("Biais de la FC moyenne\n— FC intégrée >> FC moyenne pour les intervalles")
plt.tight_layout()
plt.savefig("fig_trimp_biais_intervalles.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Séance A  (continu,     FC moy.={hr_A:.0f}) : TRIMP = {trimp_A:.1f}")
print(f"Séance B  (intervalles, FC moy.={hr_B_mean:.0f}) : TRIMP Bannister naïf = {trimp_B_avg:.1f}")
print(f"Séance B  (intervalles, FC intégrée)           : TRIMP 'réel'         = {trimp_B_cont:.1f}")


---
## 6. Sensibilité aux constantes de temps τ₁ et τ₂

Les constantes 42 j (CTL) et 7 j (ATL) sont des **valeurs par défaut**, pas des  
vérités universelles. Leur variabilité inter-individuelle est documentée.

*Source : Coggan (2023), trainingpeaks.com ; IJspp 17(5) (2022).*


In [ ]:
tss_seq = trimp_seq.copy()

configs = [
    {"tau_ctl": 42, "tau_atl": 7,  "label": "Standard (42/7)",    "ls": "-"},
    {"tau_ctl": 28, "tau_atl": 5,  "label": "Récupération rapide (28/5)",  "ls": "--"},
    {"tau_ctl": 56, "tau_atl": 10, "label": "Récupération lente  (56/10)", "ls": "-."},
]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

colors_cfg = ["#2e7bcf", "#e05c3a", "#2c9c5a"]

for cfg, col in zip(configs, colors_cfg):
    ctl, atl, tsb = simulate_ctl_atl_tsb(tss_seq, cfg["tau_ctl"], cfg["tau_atl"])
    axes[0].plot(days, ctl, color=col, lw=2, ls=cfg["ls"], label=cfg["label"])
    axes[1].plot(days, tsb, color=col, lw=2, ls=cfg["ls"], label=cfg["label"])

axes[0].set_ylabel("CTL")
axes[0].set_title("Sensibilité du CTL aux constantes de temps τ₁")
axes[0].legend()
axes[1].axhline(0, color="gray", ls=":", lw=1)
axes[1].set_xlabel("Jours")
axes[1].set_ylabel("TSB")
axes[1].set_title("Sensibilité du TSB aux constantes de temps")
axes[1].legend()
plt.tight_layout()
plt.savefig("fig_sensibilite_tau.png", dpi=150, bbox_inches="tight")
plt.show()


---
## 7. Extension data-driven — au-delà des modèles figés

Les modèles de Bannister et Coggan supposent des constantes $\tau_1, \tau_2, k_1, k_2$  
**universelles et fixes**. En pratique, elles varient selon l'athlète, la période de saison  
et le type de charge.

Deux approches data-driven complémentaires sont présentées ici :

1. **Ajustement des paramètres par optimisation** : on cale $k_1, k_2, \tau_1, \tau_2$ sur  
   des mesures de performance réelles (chrono, test de terrain).
2. **Régression locale** : on ajuste une décroissance exponentielle directement sur la  
   courbe CTL ou ATL observée pour estimer $\tau$ a posteriori sur les données personnelles.

*Pour aller plus loin : Busso (2003), Med. Sci. Sports Exerc., 35(7) ; Hellard et al. (2006) ;*  
*Série d'articles sur les modèles fitness-fatigue dans IJspp (2022).*


In [ ]:
# ── 7a. Optimisation des paramètres sur performances observées ─────────────

def model_performance(trimp_seq, k1, k2, tau1, tau2, p0=100.0):
    """Wrapper for curve_fit: returns predicted performance at observed days."""
    _, _, perf = simulate_fitness_fatigue(trimp_seq, k1, k2, tau1, tau2, p0)
    return perf


def fit_model_to_performances(trimp_seq, perf_days, perf_values,
                               p0_init=(1.0, 2.0, 45.0, 11.0)):
    """Fit Banister model parameters to measured performances.

    Parameters
    ----------
    trimp_seq : array
        Full daily TRIMP sequence.
    perf_days : array of int
        Indices (days) where performance was measured.
    perf_values : array
        Measured performance values.
    p0_init : tuple
        Initial guess (k1, k2, tau1, tau2).

    Returns fitted parameters and covariance.
    """
    def model_at_days(dummy, k1, k2, tau1, tau2):
        _, _, perf = simulate_fitness_fatigue(trimp_seq, k1, k2, tau1, tau2)
        return perf[perf_days]

    bounds = ([0, 0, 10, 3], [5, 10, 100, 30])
    popt, pcov = curve_fit(
        model_at_days,
        np.zeros(len(perf_days)),  # dummy x requis par curve_fit
        perf_values,
        p0=p0_init,
        bounds=bounds,
        maxfev=5000
    )
    return popt, pcov


# Simulation : on génère des "vraies" performances avec des paramètres connus
TRUE_PARAMS = (1.0, 2.2, 48.0, 10.0)
np.random.seed(0)
_, _, perf_true = simulate_fitness_fatigue(trimp_seq, *TRUE_PARAMS)

# On "mesure" la performance tous les 15 jours avec un peu de bruit
meas_days = np.arange(0, n_days, 15)
meas_vals = perf_true[meas_days] + np.random.normal(0, 1.5, len(meas_days))

# Ajustement
popt, _ = fit_model_to_performances(trimp_seq, meas_days, meas_vals)
print("Paramètres vrais  :", dict(zip(["k1","k2","τ1","τ2"], TRUE_PARAMS)))
print("Paramètres ajustés:", dict(zip(["k1","k2","τ1","τ2"], [f"{v:.2f}" for v in popt])))

_, _, perf_fit = simulate_fitness_fatigue(trimp_seq, *popt)

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(days, perf_true, color="#aaa", lw=1.5, label="Performance 'vraie' (paramètres connus)")
ax.plot(days, perf_fit,  color="#2e7bcf", lw=2, label="Modèle ajusté (curve_fit)")
ax.scatter(meas_days, meas_vals, color="#e05c3a", zorder=5,
           label=f"Mesures terrain (n={len(meas_days)}, bruit σ=1.5)")
ax.set_xlabel("Jours")
ax.set_ylabel("Performance (u.a.)")
ax.set_title("Ajustement data-driven des paramètres de Banister")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("fig_fit_parametres.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 7b. Estimation locale de τ sur la courbe CTL/ATL ─────────────────────

def estimate_tau_from_decay(day_start, n_rest_days, ctl_series):
    """Estimate tau from an observed CTL decay during a rest period.

    Parameters
    ----------
    day_start : int
        First day of rest (index in ctl_series).
    n_rest_days : int
        Number of consecutive rest days.
    ctl_series : array
        Observed CTL values.
    """
    t = np.arange(n_rest_days)
    y = ctl_series[day_start:day_start + n_rest_days]
    y0 = y[0]

    def exp_decay(t, tau):
        return y0 * np.exp(-t / tau)

    popt, _ = curve_fit(exp_decay, t, y, p0=[42.0], bounds=([5], [200]))
    return popt[0]


# Période de repos simulée : jours 60-69 (TSS~20, quasi-repos)
rest_start, rest_len = 60, 10
ctl_obs, _, _ = simulate_ctl_atl_tsb(trimp_seq)
tau_est = estimate_tau_from_decay(rest_start, rest_len, ctl_obs)
print(f"τ CTL estimé sur la période de repos (jours {rest_start}-{rest_start+rest_len}) : {tau_est:.1f} j")
print(f"(valeur standard utilisée : 42 j)")

# Visualisation
t_local = np.arange(rest_len)
y_obs   = ctl_obs[rest_start:rest_start + rest_len]
y_fit   = y_obs[0] * np.exp(-t_local / tau_est)
y_std   = y_obs[0] * np.exp(-t_local / 42.0)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t_local, y_obs, "o-", color="#2e7bcf", lw=2, label="CTL observé (période repos)")
ax.plot(t_local, y_fit, color="#e05c3a", lw=2, ls="--", label=f"Ajustement : τ = {tau_est:.1f} j")
ax.plot(t_local, y_std, color="#aaa",    lw=1.5, ls=":",  label="Standard : τ = 42 j")
ax.set_xlabel("Jours depuis le début du repos")
ax.set_ylabel("CTL")
ax.set_title("Estimation locale de τ par décroissance exponentielle")
ax.legend()
plt.tight_layout()
plt.savefig("fig_tau_local.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Résumé et points-clés

| Modèle | Avantage principal | Limite principale |
|---|---|---|
| TRIMP Bannister (1975) | Simple, un seul chiffre par séance | FC moyenne, coefficients génériques |
| TRIMP Edwards (1993) | Zones accessibles sans labo | Zones et coefficients arbitraires |
| TRIMP Lucia (2003) | Zones ancrées sur SV1/SV2 | Coefficients arbitraires, labo nécessaire |
| Fitness-fatigue Morton (1990) | Modélise forme ET fatigue dans le temps | Paramètres difficiles à caler, 4 inconnues |
| CTL/ATL/TSB Coggan (2010) | Lisible, implémenté partout | Constantes fixes, non validé performance |
| Data-driven (curve_fit) | Paramètres personnalisés | Nécessite mesures terrain régulières |

**Pour un trailer** : le couple CTL/TSB reste l'outil le plus accessible pour planifier  
un pic de forme sur une échéance (SaintéLyon, UTHG…). L'extension data-driven devient  
pertinente dès qu'on dispose de tests de terrain réguliers (Cooper, VMA, chrono sur boucle fixe).
